# Morteza Project: IMU and xdf file



We have a file in XDF format containing kinematic IMU data. This format, due to its sampling method, helps ensure that the recording time for each sensor is identical and prevents timing discrepancies by synchronizing the recording timestamps.

Objective: To read and analyze the file.


In [1]:
# Basic Packages
import os

# xdf reader
import pyxdf # pip install pyxdf

# Analysis Packages
import numpy as np
import pandas as pd

# Plot Packages
import matplotlib.pyplot as plt
import seaborn as sns

# Static Package
import scipy.stats as stats

### PATH of input and output

In [2]:
PATH = os.path.dirname(os.getcwd())
NAME_FOLDER_INPUT = r"data\raw\6. Mrteza Project IMU and xdf file\TEST 27 MAI"
NAME_FOLDER_OUTPUT = r"data\processed\6. Mrteza Project IMU and xdf file\TEST 27 MAI"

# Make input and output path
INPUT_PATH = os.path.join(PATH, NAME_FOLDER_INPUT)
INPUT_OUTPUT = os.path.join(PATH, NAME_FOLDER_OUTPUT)

### Read name of all file by `xdf` format

create one dataframe from name of all data   
create one column from name of dataframe   
`inf_all_sdf_data` is a dataframe from file name and name of each dataframe

In [3]:
# read name of each file as .xdf
name_all_file = [f for f in os.listdir(INPUT_PATH) if f.endswith(".xdf")]

# create datafram from name of data by .xdf
inf_all_sdf_data = pd.DataFrame(name_all_file, columns=["name_file"])

# create name of each dataframe
inf_all_sdf_data["name_df"] = (inf_all_sdf_data["name_file"].
                               str.replace(".xdf", "").
                               str.replace("-","_").
                               str.replace(" ", "_"))
inf_all_sdf_data

,name_file,name_df
0,abduction.xdf,abduction
1,flexionandextension.xdf,flexionandextension
2,joggen.xdf,joggen
3,squat.xdf,squat
4,standing.xdf,standing


### 1. Read all data

In [7]:
# fuction reader file
def path_each_file(name_each_file:str, name_each_df:str):
    """function **path_each_file**   
    give 2 parametr   
    name_each_file: .xdf   
    name_each_df: name of each dataframe
    """
    path_read = os.path.join(INPUT_PATH, name_each_file)
    globals()[name_each_df], header = pyxdf.load_xdf(path_read)
    return globals()[name_each_df]



# maping file for sensor name
maping_file = [f for f in os.listdir(INPUT_PATH) if f.endswith(".xlsx")]
PATH_MAP_SENSOR = os.path.join(INPUT_PATH, maping_file[0])
MAP_SENSOR = pd.read_excel(PATH_MAP_SENSOR, sheet_name="Tabelle1")

# create all dataframe
name_dataframe = []
for each_df in range(inf_all_sdf_data.shape[0]):
    name_each_file = inf_all_sdf_data.iloc[each_df]["name_file"]
    name_each_df = inf_all_sdf_data.iloc[each_df]["name_df"]
    path_each_file(name_each_file, name_each_df)
    
    data = globals()[name_each_df]
    valid_streams = [s for s in data if len(s['time_series']) > 0] # mask data, time_series is not empty
    
    for stream in valid_streams:
        
        # name sensor by mapping file
        name_df= stream['info']['name'][0]
        name_sensor = MAP_SENSOR[MAP_SENSOR["ID"].eq(name_df[-8:])]["Name"].iloc[0]
        #---------------------
        name_dataframe.append(f"{name_each_df}_{name_sensor}") # name of each df
        
        df = pd.DataFrame(stream['time_series']) # create df by time_series
        df.index = stream['time_stamps'] # set index by time_stamps
        df.index.name = 'timestamp' # set name of index
        df.columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']] # set name of each column
        globals()[f"{name_each_df}_{name_sensor}"] = df # create df by orginal name





Stream 1: last clock offset is statistically anomalous, truncating (see pylsl#67, liblsl#246).
Stream 1: sample count (1976) exceeds footer sample_count (1975), truncating extra samples.


### Create info IMU df

In [8]:
inf_IMU = pd.DataFrame(name_dataframe, columns=["name_dataframe"])
inf_IMU["name_sensor"] = inf_IMU["name_dataframe"].str.split("_", n=1).str[0]
inf_IMU["name_loc_sensor"] = inf_IMU["name_dataframe"].str.split("_", n=1).str[1]

In [9]:
inf_IMU["name_loc_sensor"].unique()

array(['pelvis', 'femur_l', 'tibia_l', 'calcn_l'], dtype=object)

In [ ]:
inf_IMU[inf_IMU["name_loc_sensor"].eq("pelvis")]


,name_dataframe,name_sensor,name_loc_sensor
0,abduction_pelvis,abduction,pelvis
4,flexionandextension_pelvis,flexionandextension,pelvis
9,joggen_pelvis,joggen,pelvis
12,squat_pelvis,squat,pelvis
16,standing_pelvis,standing,pelvis


In [14]:
inf_IMU.head(4)

,name_dataframe,name_sensor,name_loc_sensor
0,abduction_pelvis,abduction,pelvis
1,abduction_femur_l,abduction,femur_l
2,abduction_tibia_l,abduction,tibia_l
3,abduction_calcn_l,abduction,calcn_l


## 2. Calibration  

### Sample Data Descriptions

The five samples: `'squat.xdf'`, `'standing.xdf'`, `'abduction.xdf'`, `'flexionandextension.xdf'`, and `'joggen.xdf'` represent the following:

*   **`standing.xdf` (Static Standing):** Used for initial calibration to find the direction of gravity and the vertical axis for each sensor.
*   **`squat.xdf` (Squat Movement):** Used to better estimate and check the flexion/extension axes, specifically for the femur and tibia.
*   **`flexionandextension.xdf` (Controlled Flexion/Extension):** Used to locate the primary flexion/extension axis using the gyroscope.
*   **`abduction.xdf` (Abduction Movement):** Used to find the abduction/adduction axis and assist in determining the anatomical orientation.
*   **`joggen.xdf` (Running/Jogging Data):** The main movement dataset. This file is used after calibration to transform sensor orientation into bone orientation.

-----

- **`standing.xdf`**  
  ایستادن ثابت.  
  برای کالیبراسیون اولیه، پیدا کردن جهت گرانش و محور عمودی هر سنسور.

- **`squat.xdf`**  
  حرکت اسکوات.  
  برای بررسی و تخمین بهتر محورهای فلکشن/اکستنشن، مخصوصاً برای femur و tibia.

- **`flexionandextension.xdf`**  
  حرکت خم و راست شدن کنترل‌شده.  
  برای پیدا کردن محور اصلی **flexion/extension** با استفاده از ژیروسکوپ.

- **`abduction.xdf`**  
  حرکت دور کردن اندام از خط وسط بدن.  
  برای پیدا کردن محور **abduction/adduction** و کمک به تعیین جهت‌گیری آناتومیکی.

- **`joggen.xdf`**  
  دیتای اصلی حرکت دویدن/جاگینگ.  
  این فایل بعد از کالیبراسیون استفاده می‌شود تا orientation سنسورها به orientation استخوان تبدیل شود.

اگر بخواهی، قدم بعدی خیلی کوتاه می‌گویم **هر کدام دقیقاً برای کدام محور/ماتریس استفاده می‌شود**.

For each sensor, find a fixed transformation that maps the sensor coordinate system to the bone/anatomical segment coordinate system.

$$
R_{B \leftarrow S}
$$

$S$: Sensor frame   
$B$: Bone/anatomical frame   
Then, apply this matrix to the main `joggen` data so that the orientations are expressed in the bone frame.

---

برای هر سنسور، یک تبدیل ثابت پیدا کنی که دستگاه مختصات سنسور را به دستگاه مختصات استخوان/سگمنت آناتومیک ببرد.

یعنی برای هر سگمنت مثل pelvis, femur, tibia, calcaneus می‌خواهی این را پیدا کنی:

$$
R_{B \leftarrow S}
$$
​
$S$: فریم سنسور   
$B$: : فریم استخوان/آناتومیک   
بعد این ماتریس را روی دیتای اصلی joggen اعمال می‌کنی تا اورینتیشن‌ها در فریم استخوانی بیان شوند.

------

Logical use of each file:

- `standing` → used to define the body's vertical axis with the help of gravity
- `flexionandextension` → used to find the main flexion/extension axis
- `abduction` → used to find the abduction/adduction axis
- `squat` → used to validate or stabilize the estimation of the joint axis
- `joggen` → the main dataset that will be transformed later

----- 
استفاده منطقی از هر کدام:   
standing → برای تعریف محور عمودی بدن با کمک گرانش   
flexionandextension → برای پیدا کردن محور اصلی فلکشن/اکستنشن   
abduction → برای پیدا کردن محور ابداکشن/اداکشن   
squat → برای اعتبارسنجی یا پایدارتر کردن تخمین محور مفصلی   
joggen → دیتای اصلی که بعداً transform می‌شود   

### Final Output to Build

For each segment:

- `R_pelvis_BS`
- `R_femur_BS`
- `R_tibia_BS`
- `R_calcn_BS`

These will later be used for each time step of `joggen`:

$$
R_{\text{global} \leftarrow \text{bone}}(t)
=
R_{\text{global} \leftarrow \text{sensor}}(t)
\;
R_{\text{sensor} \leftarrow \text{bone}}
$$

If you have:

$$
R_{B \leftarrow S}
$$

then:

$$
R_{\text{global} \leftarrow \text{bone}}(t)
=
R_{\text{global} \leftarrow \text{sensor}}(t)
\;
R_{B \leftarrow S}^{T}
$$

Or if you work with the opposite definition:

$$
R_{\text{global} \leftarrow \text{bone}}(t)
=
R_{\text{global} \leftarrow \text{sensor}}(t)
\;
R_{S \leftarrow B}
$$

So, first, you must keep the matrix convention fixed from the beginning.


In [28]:
# 1st normalize

def normalize(v:float):
    """function **normalize**    
    give one vector and normalize   
    1st: create an array by numpy    
    2rd: This function calculates the norm (length or magnitude) of a vector.  
    This normalizes a vector, meaning it scales the vector so that its length becomes 1.     
    3rd: np.asarray(v) / np.linalg.norm(v)
    """
    v = np.asarray(v)
    n = np.linalg.norm(v)
    if n < 1e-12: # if value / 0 is error
        raise ValueError("Zero vector")
    return v / n


# 2rd: principal_axis
def principal_axis(X):
    """
    If you rotate the sensor around an axis, this function gives you that fixed axis in the sensor space.    
    It is excellent for finding the Flexion/Extension axis.   

    In biomechanics, the sensor is not mounted perfectly parallel to the anatomical axes of the body.

    Therefore, we perform PCA to:   
    - Find the true joint axis: determine around which axis the sensor actually rotates (e.g., the knee flexion/extension axis).
    - Reduce noise: remove the effect of small movements in other directions and keep only the primary direction of motion.
    - Become independent of sensor placement: even if the sensor is attached slightly misaligned, this method finds the rotation axis in the sensor’s coordinate system so it can later be calibrated.

    X: shape (N, 3)   
    returns dominant unit axis
    """
    X = np.asarray(X, dtype=float)
    X = X - X.mean(axis=0, keepdims=True) # axis= row
    C = np.cov(X.T) # covariance matrix
    # Eigenvectors → principal directions of the data
    # Eigenvalues → amount of variance in that direction
    eigvals, eigvecs = np.linalg.eigh(C)
    # It finds the direction in which the largest variations (motion) occurred.
    v = eigvecs[:, np.argmax(eigvals)]
    return normalize(v)


# 3rd: orthogonalize
def orthogonalize(v, ref):
    ref = normalize(ref)
    v = np.asarray(v, dtype=float)
    v = v - np.dot(v, ref) * ref
    return normalize(v)


# 4th: build_frame
def build_frame(z_axis, x_axis):
    """
    z_axis: long axis
    x_axis: functional axis
    returns R_S_B with columns [x, y, z]
    """
    z = normalize(z_axis)
    x = orthogonalize(x_axis, z)
    y = normalize(np.cross(z, x))
    x = normalize(np.cross(y, z))
    return np.column_stack([x, y, z])


# 5th : standing_long_axis
# Assumption: During standing, the body is approximately stationary and the mean acceleration ≈ gravity.
def standing_long_axis(acc_df):
    # acc_df columns: ax ay az
    a_mean = acc_df[['ax', 'ay', 'az']].mean().values
    g_axis = normalize(a_mean)
    return g_axis

# 6th: gyro_axis
def gyro_axis(gyro_df):
    X = gyro_df[['gx', 'gy', 'gz']].values
    return principal_axis(X)



In [4]:
# path of one data
path_read = os.path.join(INPUT_PATH, inf_all_sdf_data["name_file"].iloc[0])

# raed data
data, header = pyxdf.load_xdf(path_read)

In [5]:
header

{'info': defaultdict(list,
             {'version': ['1.0'], 'datetime': ['2026-05-27T17:17:06+0200']})}

In [11]:
name_each_Xsens = []
for i in range(len(data)):
    name_each_Xsens.append(data[i]["info"]["name"])

name_each_Xsens

[['Xsens_MTw2_00B4D0C5'],
 ['Xsens_MTw2_00B4D0C8'],
 ['Xsens_MTw2_00B4D0BE'],
 ['Xsens_MTw2_00B4D0D0']]

In [13]:
data[0]['info']['desc'][0]['channels'][0]['channel']

[defaultdict(list, {'label': ['qw']}),
 defaultdict(list, {'label': ['qx']}),
 defaultdict(list, {'label': ['qy']}),
 defaultdict(list, {'label': ['qz']}),
 defaultdict(list, {'label': ['ax']}),
 defaultdict(list, {'label': ['ay']}),
 defaultdict(list, {'label': ['az']}),
 defaultdict(list, {'label': ['gx']}),
 defaultdict(list, {'label': ['gy']}),
 defaultdict(list, {'label': ['gz']}),
 defaultdict(list, {'label': ['mx']}),
 defaultdict(list, {'label': ['my']}),
 defaultdict(list, {'label': ['mz']}),
 defaultdict(list, {'label': ['packet_counter']}),
 defaultdict(list, {'label': ['sample_time_fine']})]

In [125]:
data[0].keys(), data[6].keys()

(dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values']),
 dict_keys(['info', 'footer', 'time_series', 'time_stamps', 'clock_times', 'clock_values']))

In [ ]:
stream = data[0]
df = pd.DataFrame(stream['time_series'])
df.index = stream['time_stamps']
df.index.name = 'timestamp'
df.columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']]
df

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.472544,0.099020,0.005810,-0.002649,-0.995065,-0.090631,0.075291,9.776200,-0.008962,0.006068,-0.002955,12303.0,0.0
229501.482596,0.099021,0.005805,-0.002610,-0.995065,-0.083169,0.051323,9.751882,-0.006871,0.003363,-0.001430,12304.0,0.0
229501.492647,0.099018,0.005803,-0.002555,-0.995066,-0.091469,0.073681,9.776204,-0.010151,0.004301,-0.001368,12305.0,0.0
229501.502698,0.099004,0.005797,-0.002484,-0.995067,-0.101624,0.094862,9.751747,-0.013208,0.003712,-0.004408,12306.0,0.0
229501.512749,0.098985,0.005792,-0.002421,-0.995069,-0.115895,0.062052,9.752830,-0.011709,0.003993,-0.004440,12307.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
229515.453850,0.083752,0.011817,-0.001658,-0.996415,-0.184272,0.071303,9.766708,0.003579,0.005374,-0.003182,13702.0,0.0
229515.463901,0.083762,0.011826,-0.001646,-0.996414,-0.172329,0.103592,9.766027,-0.001278,0.005996,0.001436,13703.0,0.0
229515.473952,0.083761,0.011856,-0.001644,-0.996414,-0.175751,0.049081,9.767643,0.001122,0.009563,-0.001724,13704.0,0.0


In [151]:
for i in [0, 6]:
    s = data[i]
    print(f"--- Stream {i} ---")
    print(f"Name: {s['info']['name']}")
    print(f"Type: {s['info']['type']}")
    print(f"Channel Count: {s['info']['channel_count']}")
    print(f"Series shape: {len(s['time_series'])}")
    print(f"Timestamps shape: {len(s['time_stamps'])}")
    print("-" * 20)

--- Stream 0 ---
Name: ['Xsens_MTw2_00B4D0C8']
Type: ['IMU']
Channel Count: ['12']
Series shape: 1396
Timestamps shape: 1396
--------------------
--- Stream 6 ---
Name: ['Xsens_MTw2_00B4D0C8']
Type: ['IMU']
Channel Count: ['12']
Series shape: 0
Timestamps shape: 0
--------------------


In [169]:
valid_streams = [s for s in data if len(s['time_series']) > 0]
name_dataframe = []
for stream in valid_streams:
    print(f"name dataframe: {stream['info']['name']}")
    name_dataframe.append(stream['info']['name'])
    globals()[stream['info']['name'][0]] = pd.DataFrame(stream['time_series'])
    globals()[stream['info']['name'][0]].index = stream['time_stamps']
    globals()[stream['info']['name'][0]].index.name = 'timestamp'
    globals()[stream['info']['name'][0]].columns = [ch['label'][0] for ch in stream['info']['desc'][0]['channels'][0]['channel']]
   

name dataframe: ['Xsens_MTw2_00B4D0C8']
name dataframe: ['Xsens_MTw2_00B4D0BE']
name dataframe: ['Xsens_MTw2_00B4D0C5']
name dataframe: ['Xsens_MTw2_00B4D0C4']
name dataframe: ['Xsens_MTw2_00B4D0D0']
name dataframe: ['Xsens_MTw2_00B4D0BF']


In [170]:
name_dataframe

[['Xsens_MTw2_00B4D0C8'],
 ['Xsens_MTw2_00B4D0BE'],
 ['Xsens_MTw2_00B4D0C5'],
 ['Xsens_MTw2_00B4D0C4'],
 ['Xsens_MTw2_00B4D0D0'],
 ['Xsens_MTw2_00B4D0BF']]

In [178]:
Xsens_MTw2_00B4D0C8.head(2)

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.472544,0.099020,0.005810,-0.002649,-0.995065,-0.090631,0.075291,9.776200,-0.008962,0.006068,-0.002955,12303.0,0.0
229501.482596,0.099021,0.005805,-0.002610,-0.995065,-0.083169,0.051323,9.751882,-0.006871,0.003363,-0.001430,12304.0,0.0


In [ ]:
Xsens_MTw2_00B4D0BE.head(2)

,qw,qx,qy,qz,ax,ay,az,gx,gy,gz,packet_counter,sample_time_fine
timestamp,,,,,,,,,,,,
229501.461992,0.334465,0.003907,-0.000751,-0.942400,-0.064216,0.030809,9.730543,0.001064,0.005699,-0.002444,12303.0,0.0
229501.472052,0.334455,0.003904,-0.000747,-0.942403,-0.056171,0.045107,9.700230,0.000548,0.002237,-0.005045,12304.0,0.0


### Reading all data streams simultaneously.   
Using the `inf_all_sdf_data` DataFrame, we read the individual dataframes and create a specific dataset for each, named accordingly.


In [43]:
for file in inf_all_sdf_data["name_file"]:
    print(file)

sub-P001_ses-S001_task-Default_run-001_eeg.xdf
